
## Global Per‑Band Stats (window_b)

This notebook computes **global per‑band statistics** across all GeoTIFFs pointed to by the `window_b` column in catalog csv files:

- **Exact** mean & standard deviation 
- **1st, 2nd, 98th, 99th percentiles** 
- Skips `nodata` and NaNs.

**Outputs**
- `global_band_stats_ftw.csv`, `global_band_stats_ftw_mappingafrica_combined.csv` and `global_band_stats_mappingafrica.csv`


In [ ]:
# Imports

import math
import os
import json
import numpy as np
import pandas as pd
import rasterio
from collections import defaultdict
from __future__ import annotations
from typing import Dict, Iterable, Iterator, List, Tuple
from rasterio.io import DatasetReader
from tqdm import tqdm

# Display settings

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

In [ ]:
# Configuration

input_catalog_path: str = (
    "/home/airg/gessuman/projects/ftw-mappingafrica-integration/data/"
    "ftw-mappingafrica-combined-catalog.csv"
)
path_column_name: str = "window_b"                     # column with TIFF paths
root_data_dir: str = "/home/airg/data/labels/cropland/"  # prefix for relative paths

histogram_bin_count: int = 4096                        # bins for percentiles
max_files_to_process: int = 0                          # 0 = all files; set to 1 for a quick test

output_csv_path: str = (
    "/home/airg/gessuman/projects/ftw-mappingafrica-integration/data/"
    "global_band_stats_ftw_mappingafrica_combined.csv"
)
output_json_path: str = (
    "/home/airg/gessuman/projects/ftw-mappingafrica-integration/data/"
    "global_band_stats_ftw_mappingafrica_combined.json"
)

In [ ]:
# I/O helper

def resolve_path(path_str: str, root_dir: str) -> str:
    """Resolve a relative file path.

    If `path_str` is absolute, return it unchanged; otherwise return `root_dir/path_str`.

    Args:
        path_str: A file path from the catalog.
        root_dir: Root directory to prefix when `path_str` is relative.

    Returns:
        The absolute path to the file.
    """
    return path_str if os.path.isabs(path_str) else os.path.join(root_dir, path_str)

In [ ]:
# Function for welford algorithm

def welford_merge_update(
    sample_values: np.ndarray,
    running_state: Tuple[int, float, float],
) -> Tuple[int, float, float]:
    """Merge a chunk of values into a running Welford (n, mean, M2) state.

    Implements the parallel/merge form of Welford’s algorithm for numerically
    stable streaming mean and sample variance.

    Args:
        sample_values: 1-D array of finite numeric samples.
        running_state: Tuple of (count n, running_mean, running_M2).

    Returns:
        Updated (n, mean, M2) after merging `sample_values`.
    """
    n_running, mean_running, M2_running = running_state
    if sample_values.size == 0:
        return running_state

    n_chunk: int = int(sample_values.size)
    mean_chunk: float = float(np.mean(sample_values))
    M2_chunk: float = float(np.sum((sample_values - mean_chunk) ** 2))

    mean_delta: float = mean_chunk - mean_running
    n_total: int = n_running + n_chunk

    if n_total == 0:
        return running_state

    mean_total: float = mean_running + mean_delta * (n_chunk / n_total)
    M2_total: float = (
        M2_running + M2_chunk + (mean_delta**2) * (n_running * n_chunk / n_total)
    )
    return n_total, mean_total, M2_total

In [ ]:
# Function for iterating imagery in blocks

def iterate_valid_blocks(
    dataset: DatasetReader,
    band_indices: List[int],
) -> Iterator[Tuple[int, np.ndarray]]:
    """Stream valid pixel values per band over the dataset's internal windows.

    Uses rasterio's `block_windows` to read small windows (tiles/strips) so memory
    stays bounded. Returns only finite (non-nodata, non-NaN, non-Inf) values.

    Args:
        dataset: Open rasterio dataset.
        band_indices: 1-based band indices to iterate (e.g., [1,2,3,4]).

    Yields:
        (band_index, values_1d) for each band present in each window.
    """
    read_kwargs = dict(masked=True, boundless=False)
    for _, window in dataset.block_windows(band_indices[0]):
        window_data = dataset.read(band_indices, window=window, **read_kwargs)  # (B, h, w)
        for local_idx, band_index in enumerate(band_indices):
            band_data = window_data[local_idx]
            if np.ma.isMaskedArray(band_data):
                values = band_data.compressed()
            else:
                values = band_data.ravel()
            values = values[np.isfinite(values)]
            if values.size:
                yield band_index, values

In [ ]:
# Function to generate percentile from hist

def percentile_from_histogram(
    histogram_counts: np.ndarray,
    bin_edges: np.ndarray,
    percentile_q: float,
) -> float:
    """Estimate a percentile from histogram counts via linear interpolation.

    Args:
        histogram_counts: 1-D array of non-negative counts per bin (length = B).
        bin_edges: 1-D array of bin boundaries (length = B+1), ascending.
        percentile_q: Percentile in [0, 100].

    Returns:
        Interpolated percentile value. Returns NaN if histogram is empty.
    """
    total_count = np.sum(histogram_counts)
    if total_count == 0:
        return float("nan")

    target_rank = (percentile_q / 100.0) * total_count
    cumulative_counts = np.cumsum(histogram_counts)

    bin_index = int(np.searchsorted(cumulative_counts, target_rank, side="left"))
    bin_index = int(np.clip(bin_index, 0, len(histogram_counts) - 1))

    count_before = 0 if bin_index == 0 else cumulative_counts[bin_index - 1]
    count_in_bin = int(histogram_counts[bin_index])
    if count_in_bin == 0:
        # Degenerate case: fall back to left edge
        return float(bin_edges[bin_index])

    fraction_within_bin = (target_rank - count_before) / count_in_bin
    bin_left = float(bin_edges[bin_index])
    bin_right = float(bin_edges[bin_index + 1])
    return float(bin_left + fraction_within_bin * (bin_right - bin_left))


In [ ]:
# Load catalog and preview

catalog_dataframe: pd.DataFrame = pd.read_csv(input_catalog_path)
display(catalog_dataframe.head())

path_series: pd.Series = catalog_dataframe[path_column_name].dropna().astype(str)

print(f"Total rows in catalog: {len(catalog_dataframe)}")
print(f"Non-null paths in '{path_column_name}': {len(path_series)}")

print("\nSample raw → resolved paths:")
for raw_path in path_series.head(5):
    resolved = resolve_path(raw_path, root_data_dir)
    print(f"  {raw_path}  ->  {resolved}")

In [ ]:
# Pass 1 — global min/max + exact mean/std (streaming)

all_file_paths: List[str] = path_series.tolist()
if max_files_to_process and max_files_to_process > 0:
    all_file_paths = all_file_paths[:max_files_to_process]

global_min_per_band: Dict[int, float] = defaultdict(lambda: np.inf)
global_max_per_band: Dict[int, float] = defaultdict(lambda: -np.inf)
welford_state_per_band: Dict[int, Tuple[int, float, float]] = defaultdict(
    lambda: (0, 0.0, 0.0)
)

detected_band_count: int | None = None
processed_file_count: int = 0
missing_file_count: int = 0

for raw_path in tqdm(
    all_file_paths,
    desc="Pass 1 (min/max + mean/std)",
    unit="file",
    mininterval=30,  # reduce Jupyter IOPub spam
):
    tif_path = resolve_path(raw_path, root_data_dir)
    if not os.path.exists(tif_path):
        missing_file_count += 1
        tqdm.write(f"[WARN] Missing file, skipping: {tif_path}")
        continue

    with rasterio.open(tif_path) as ds:
        band_indices = list(range(1, ds.count + 1))
        if detected_band_count is None:
            detected_band_count = ds.count
        elif ds.count != detected_band_count:
            raise RuntimeError(
                f"Inconsistent band count: {tif_path} has {ds.count}, "
                f"expected {detected_band_count}"
            )

        for band_index, values in iterate_valid_blocks(ds, band_indices):
            # Update global min/max
            global_min_per_band[band_index] = min(
                global_min_per_band[band_index], float(np.min(values))
            )
            global_max_per_band[band_index] = max(
                global_max_per_band[band_index], float(np.max(values))
            )
            # Update Welford state (exact mean/std)
            welford_state_per_band[band_index] = welford_merge_update(
                values.astype(np.float64, copy=False),
                welford_state_per_band[band_index],
            )

    processed_file_count += 1

if processed_file_count == 0:
    raise RuntimeError("No files were processed. Check catalog paths and ROOT dir.")

print(
    f"""
Pass 1 complete:
- processed files: {processed_file_count}
- missing files:   {missing_file_count}
- bands detected:  {detected_band_count}
"""
)
for band_index in range(1, detected_band_count + 1):
    print(
        f"Band {band_index}: min={global_min_per_band[band_index]}, "
        f"max={global_max_per_band[band_index]}"
    )


In [ ]:
# Pass 2 — build global histograms then percentiles

global_histogram_per_band: Dict[int, np.ndarray] = {}
global_edges_per_band: Dict[int, np.ndarray] = {}

for band_index in range(1, detected_band_count + 1):
    vmin = global_min_per_band[band_index]
    vmax = global_max_per_band[band_index]

    if not np.isfinite(vmin) or not np.isfinite(vmax):
        vmin, vmax = 0.0, 1.0
    if math.isclose(vmin, vmax):
        vmax = vmin + 1e-9  # avoid zero-width range

    bin_edges = np.linspace(vmin, vmax, histogram_bin_count + 1, dtype=np.float64)
    global_edges_per_band[band_index] = bin_edges
    global_histogram_per_band[band_index] = np.zeros(
        histogram_bin_count, dtype=np.int64
    )

for raw_path in tqdm(
    all_file_paths,
    desc="Pass 2 (global histograms)",
    unit="file",
    mininterval=30,  # reduce Jupyter IOPub spam
):
    tif_path = resolve_path(raw_path, root_data_dir)
    if not os.path.exists(tif_path):
        continue

    with rasterio.open(tif_path) as ds:
        band_indices = list(range(1, ds.count + 1))
        for band_index, values in iterate_valid_blocks(ds, band_indices):
            bin_edges = global_edges_per_band[band_index]
            left_edge, right_edge = bin_edges[0], bin_edges[-1]

            # Clip to global range (rare, but keeps counts bounded)
            values = values[(values >= left_edge) & (values <= right_edge)]
            if values.size == 0:
                continue

            hist_counts, _ = np.histogram(values, bins=bin_edges)
            global_histogram_per_band[band_index] += hist_counts

print("Pass 2 complete.")


In [ ]:
# Finalize global stats and save

result_records: List[Dict[str, float | int]] = []
requested_percentiles: List[int] = [1, 2, 98, 99]

for band_index in range(1, detected_band_count + 1):
    n_obs, mean_value, M2_value = welford_state_per_band[band_index]
    std_value = math.sqrt(M2_value / (n_obs - 1)) if n_obs > 1 else float("nan")

    bin_edges = global_edges_per_band[band_index]
    band_hist = global_histogram_per_band[band_index]

    percentile_values = {
        f"p{q}": percentile_from_histogram(band_hist, bin_edges, q)
        for q in requested_percentiles
    }

    record = {
        "band": band_index,
        "count": int(n_obs),
        "mean": mean_value,
        "std": std_value,
        "min": float(global_min_per_band[band_index]),
        "max": float(global_max_per_band[band_index]),
        **percentile_values,
    }
    result_records.append(record)

global_stats_df: pd.DataFrame = (
    pd.DataFrame(result_records).sort_values("band").reset_index(drop=True)
)
display(global_stats_df)

# Persist results
global_stats_df.to_csv(output_csv_path, index=False)
with open(output_json_path, "w") as f_out:
    json.dump(result_records, f_out, indent=2)

print(f"Wrote: {output_csv_path} and {output_json_path}")


### FTW - MAPPING AFRICA COMBINED

In [9]:
# import math
# import os
# from collections import defaultdict

# import numpy as np
# import pandas as pd
# import rasterio
# from tqdm import tqdm
# import json

# pd.set_option("display.max_rows", 200)
# pd.set_option("display.max_columns", 200)
# pd.set_option("display.width", 160)

# # ========================
# # Configuration
# # ========================
# CSV_PATH = "/home/airg/gessuman/projects/ftw-mappingafrica-integration/data/ftw-mappingafrica-combined-catalog.csv"
# COLUMN   = "window_b"                         # column with tif paths
# ROOT_DIR = "/home/airg/data/labels/cropland/" # base folder for relative paths

# BINS = 4096                                   # histogram bins (percentile resolution)
# MAX_FILES = 0                                 # 0 = all files; set to 1 for a quick smoke test

# # ========================
# # Preview CSV (no path-shape assumptions)
# # ========================
# df = pd.read_csv(CSV_PATH)
# display(df.head())

# # Pull non-null paths as strings
# rel = df[COLUMN].dropna().astype(str)

# print(f"Total rows in CSV: {len(df)}")
# print(f"Non-null paths in '{COLUMN}': {len(rel)}")

# # Dynamic resolution: absolute paths kept; relative paths get ROOT_DIR prefixed
# def resolve_path(p: str, root: str = "") -> str:
#     """If p is absolute, return p; else join with root."""
#     return p if os.path.isabs(p) else os.path.join(root, p)

# print("\nSample raw -> resolved paths:")
# for p in rel.head(5):
#     rp = resolve_path(p, ROOT_DIR)
#     print(f"  {p}  ->  {rp}")

# # ========================
# # Helpers (streaming stats + percentiles)
# # ========================
# def welford_update(x, state):
#     n, mean, M2 = state
#     if x.size == 0:
#         return n, mean, M2
#     n_x = x.size
#     mean_x = float(np.mean(x))
#     M2_x = float(np.sum((x - mean_x) ** 2))
#     delta = mean_x - mean
#     n_new = n + n_x
#     mean_new = mean + delta * (n_x / n_new) if n_new > 0 else mean
#     M2_new = M2 + M2_x + delta**2 * (n * n_x / n_new) if n_new > 0 else M2
#     return n_new, mean_new, M2_new

# def iter_valid_blocks(ds, bands):
#     """Yield (band_index, 1D finite values) for each internal block window."""
#     read_kwargs = dict(masked=True, boundless=False)
#     for _, window in ds.block_windows(bands[0]):
#         arr = ds.read(bands, window=window, **read_kwargs)  # (B, h, w)
#         for i, b in enumerate(bands):
#             band_vals = arr[i]
#             if np.ma.isMaskedArray(band_vals):
#                 vals = band_vals.compressed()
#             else:
#                 vals = band_vals.ravel()
#             vals = vals[np.isfinite(vals)]
#             if vals.size:
#                 yield b, vals

# def percentile_from_histogram(hist, edges, q):
#     total = np.sum(hist)
#     if total == 0:
#         return np.nan
#     target = q / 100.0 * total
#     c = np.cumsum(hist)
#     idx = np.searchsorted(c, target, side="left")
#     idx = int(np.clip(idx, 0, len(hist) - 1))
#     c_before = 0 if idx == 0 else c[idx - 1]
#     bin_count = hist[idx]
#     if bin_count == 0:
#         return float(edges[idx])
#     frac = (target - c_before) / bin_count
#     return float(edges[idx] + frac * (edges[idx + 1] - edges[idx]))

# # ========================
# # Pass 1: global min/max + exact mean/std (streaming)
# # ========================
# paths = rel.tolist()
# if MAX_FILES and MAX_FILES > 0:
#     paths = paths[:MAX_FILES]

# band_min = defaultdict(lambda: np.inf)
# band_max = defaultdict(lambda: -np.inf)
# welford_state = defaultdict(lambda: (0, 0.0, 0.0))  # (n, mean, M2)
# n_bands_global = None

# processed = 0
# missing = 0

# for p in tqdm(paths, desc="Pass 1 (min/max + mean/std)", unit="file"):
#     tif = resolve_path(p, ROOT_DIR)
#     if not os.path.exists(tif):
#         missing += 1
#         tqdm.write(f"[WARN] Missing file, skipping: {tif}")
#         continue

#     with rasterio.open(tif) as ds:
#         bands = list(range(1, ds.count + 1))
#         if n_bands_global is None:
#             n_bands_global = ds.count
#         elif ds.count != n_bands_global:
#             raise RuntimeError(f"Inconsistent band count: {tif} has {ds.count}, expected {n_bands_global}")

#         for b, vals in iter_valid_blocks(ds, bands):
#             band_min[b] = min(band_min[b], float(np.min(vals)))
#             band_max[b] = max(band_max[b], float(np.max(vals)))
#             welford_state[b] = welford_update(vals.astype(np.float64, copy=False), welford_state[b])

#     processed += 1

# if processed == 0:
#     raise RuntimeError("No files were processed. Check CSV paths and ROOT_DIR.")

# print(f"""
# Pass 1 complete:
# - processed files: {processed}
# - missing files:   {missing}
# - bands detected:  {n_bands_global}
# """)
# for b in range(1, n_bands_global + 1):
#     print(f"Band {b}: min={band_min[b]}, max={band_max[b]}")

# # ========================
# # Pass 2: build global histograms → percentiles
# # ========================
# histograms = {}
# edges_map = {}
# for b in range(1, n_bands_global + 1):
#     vmin = band_min[b]
#     vmax = band_max[b]
#     if not np.isfinite(vmin) or not np.isfinite(vmax):
#         vmin, vmax = 0.0, 1.0
#     if math.isclose(vmin, vmax):
#         vmax = vmin + 1e-9
#     edges = np.linspace(vmin, vmax, BINS + 1, dtype=np.float64)
#     histograms[b] = np.zeros(BINS, dtype=np.int64)
#     edges_map[b] = edges

# for p in tqdm(paths, desc="Pass 2 (global histograms)", unit="file"):
#     tif = resolve_path(p, ROOT_DIR)
#     if not os.path.exists(tif):
#         continue
#     with rasterio.open(tif) as ds:
#         bands = list(range(1, ds.count + 1))
#         for b, vals in iter_valid_blocks(ds, bands):
#             edges = edges_map[b]
#             vmin, vmax = edges[0], edges[-1]
#             vals = vals[(vals >= vmin) & (vals <= vmax)]
#             if vals.size == 0:
#                 continue
#             h, _ = np.histogram(vals, bins=edges)
#             histograms[b] += h

# print("Pass 2 complete.")

# # ========================
# # Finalize global stats and save
# # ========================
# records = []
# qs = [1, 2, 98, 99]
# for b in range(1, n_bands_global + 1):
#     n, mean, M2 = welford_state[b]
#     std = math.sqrt(M2 / (n - 1)) if n > 1 else float("nan")
#     edges = edges_map[b]
#     hist = histograms[b]
#     pct = {f"p{q}": percentile_from_histogram(hist, edges, q) for q in qs}
#     rec = {
#         "band": b,
#         "count": int(n),
#         "mean": mean,
#         "std": std,
#         "min": float(band_min[b]),
#         "max": float(band_max[b]),
#         **pct,
#     }
#     records.append(rec)

# stats_df = pd.DataFrame(records).sort_values("band").reset_index(drop=True)
# display(stats_df)

# out_csv  = "/home/airg/gessuman/projects/ftw-mappingafrica-integration/data/global_band_stats_ftw_mappingafrica_combined.csv"
# out_json = "/home/airg/gessuman/projects/ftw-mappingafrica-integration/data/global_band_stats_ftw_mappingafrica_combined.json"
# stats_df.to_csv(out_csv, index=False)
# with open(out_json, "w") as f:
#     json.dump(records, f, indent=2)

# print(f"Wrote: {out_csv} and {out_json}")


,name,dataset,version,country,x,y,fld_prop,nonfld_prop,null_prop,window_a,window_b,mask,split
0,g0_0000002048-0000010240,ftw,1.0.0,kenya,34.573119,0.600720,0.000519,0.0,0.999481,ftw/kenya/s2_images/window_a/g0_0000002048-000...,ftw/kenya/s2_images/window_b/g0_0000002048-000...,ftw/kenya/label_masks/semantic_3class/g0_00000...,test
1,g0_0000001024-0000010752,ftw,1.0.0,kenya,34.586902,0.628378,0.000061,0.0,0.999939,ftw/kenya/s2_images/window_a/g0_0000001024-000...,ftw/kenya/s2_images/window_b/g0_0000001024-000...,ftw/kenya/label_masks/semantic_3class/g0_00000...,train
2,g0_0000001024-0000010240,ftw,1.0.0,kenya,34.573119,0.628378,0.001083,0.0,0.998917,ftw/kenya/s2_images/window_a/g0_0000001024-000...,ftw/kenya/s2_images/window_b/g0_0000001024-000...,ftw/kenya/label_masks/semantic_3class/g0_00000...,train
3,g0_0000003072-0000008704,ftw,1.0.0,kenya,34.531677,0.573153,0.001129,0.0,0.998871,ftw/kenya/s2_images/window_a/g0_0000003072-000...,ftw/kenya/s2_images/window_b/g0_0000003072-000...,ftw/kenya/label_masks/semantic_3class/g0_00000...,train
4,g0_0000001024-0000009728,ftw,1.0.0,kenya,34.559290,0.628378,0.000916,0.0,0.999084,ftw/kenya/s2_images/window_a/g0_0000001024-000...,ftw/kenya/s2_images/window_b/g0_0000001024-000...,ftw/kenya/label_masks/semantic_3class/g0_00000...,train


Total rows in CSV: 109560
Non-null paths in 'window_b': 109560

Sample raw -> resolved paths:
  ftw/kenya/s2_images/window_b/g0_0000002048-0000010240.tif  ->  /home/airg/data/labels/cropland/ftw/kenya/s2_images/window_b/g0_0000002048-0000010240.tif
  ftw/kenya/s2_images/window_b/g0_0000001024-0000010752.tif  ->  /home/airg/data/labels/cropland/ftw/kenya/s2_images/window_b/g0_0000001024-0000010752.tif
  ftw/kenya/s2_images/window_b/g0_0000001024-0000010240.tif  ->  /home/airg/data/labels/cropland/ftw/kenya/s2_images/window_b/g0_0000001024-0000010240.tif
  ftw/kenya/s2_images/window_b/g0_0000003072-0000008704.tif  ->  /home/airg/data/labels/cropland/ftw/kenya/s2_images/window_b/g0_0000003072-0000008704.tif
  ftw/kenya/s2_images/window_b/g0_0000001024-0000009728.tif  ->  /home/airg/data/labels/cropland/ftw/kenya/s2_images/window_b/g0_0000001024-0000009728.tif


Pass 1 (min/max + mean/std): 100%|████████████████████████████████████████████| 109560/109560 [30:15<00:00, 60.36file/s]



Pass 1 complete:
- processed files: 109560
- missing files:   0
- bands detected:  4

Band 1: min=-11660.0, max=65534.0
Band 2: min=-123.0, max=65534.0
Band 3: min=-139.0, max=65534.0
Band 4: min=0.0, max=65534.0


Pass 2 (global histograms): 100%|█████████████████████████████████████████████| 109560/109560 [53:21<00:00, 34.22file/s]


Pass 2 complete.


,band,count,mean,std,min,max,p1,p2,p98,p99
0,1,7180021938,1017.844732,750.929809,-11660.0,65534.0,103.567080,131.695489,2788.295210,3090.278028
1,2,7180021938,925.609123,577.084830,-123.0,65534.0,170.383521,210.057891,2097.337713,2301.587091
2,3,7180021938,637.796861,527.734345,-139.0,65534.0,79.827545,101.601444,1691.520638,1863.869291
3,4,7180021938,2881.766788,958.208609,0.0,65534.0,582.477473,1137.495041,5114.564284,5516.382496


Wrote: /home/airg/gessuman/projects/ftw-mappingafrica-integration/data/global_band_stats_ftw_mappingafrica_combined.csv and /home/airg/gessuman/projects/ftw-mappingafrica-integration/data/global_band_stats_ftw_mappingafrica_combined.json


In [ ]:
from __future__ import annotations

import json
import math
import os
from collections import defaultdict
from typing import Dict, Iterable, Iterator, List, Sequence, Tuple

import numpy as np
import pandas as pd
import rasterio
from rasterio.io import DatasetReader
from tqdm import tqdm

# ------------------------------
# Display (Jupyter)
# ------------------------------
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# ==============================
# Configuration (FTW catalog)
# ==============================
ftw_catalog_path: str = (
    "/home/airg/gessuman/projects/ftw-mappingafrica-integration/data/ftw-catalog.csv"
)
root_data_dir: str = "/home/airg/data/labels/cropland/"   # prefix for relative paths

# Compute with 12-bit friendly bin count (good accuracy for percentiles)
histogram_bin_count: int = 4096
max_files_to_process: int = 0   # 0 = all; set 1..N to smoke-test

# ==============================
# Helpers (I/O, streaming stats, histogram percentiles)
# ==============================
def resolve_path(path_str: str, root_dir: str) -> str:
    """Resolve a possibly relative path against `root_dir`."""
    return path_str if os.path.isabs(path_str) else os.path.join(root_dir, path_str)


def welford_merge_update(
    sample_values: np.ndarray,
    running_state: Tuple[int, float, float],
) -> Tuple[int, float, float]:
    """Merge a chunk of values into a Welford (n, mean, M2) state."""
    n_running, mean_running, M2_running = running_state
    if sample_values.size == 0:
        return running_state

    n_chunk: int = int(sample_values.size)
    mean_chunk: float = float(np.mean(sample_values))
    M2_chunk: float = float(np.sum((sample_values - mean_chunk) ** 2))

    mean_delta: float = mean_chunk - mean_running
    n_total: int = n_running + n_chunk
    if n_total == 0:
        return running_state

    mean_total: float = mean_running + mean_delta * (n_chunk / n_total)
    M2_total: float = M2_running + M2_chunk + (mean_delta**2) * (n_running * n_chunk / n_total)
    return n_total, mean_total, M2_total


def iterate_valid_blocks(
    dataset: DatasetReader,
    band_indices: List[int],
) -> Iterator[Tuple[int, np.ndarray]]:
    """Yield (band_index, 1D finite values) from each internal raster window."""
    read_kwargs = dict(masked=True, boundless=False)
    for _, window in dataset.block_windows(band_indices[0]):
        window_data = dataset.read(band_indices, window=window, **read_kwargs)  # (B, h, w)
        for local_idx, band_index in enumerate(band_indices):
            band_data = window_data[local_idx]
            values = band_data.compressed() if np.ma.isMaskedArray(band_data) else band_data.ravel()
            values = values[np.isfinite(values)]
            if values.size:
                yield band_index, values


def percentile_from_histogram(
    histogram_counts: np.ndarray,
    bin_edges: np.ndarray,
    percentile_q: float,
) -> float:
    """Estimate percentile from histogram via linear interpolation inside the bin."""
    total_count = np.sum(histogram_counts)
    if total_count == 0:
        return float("nan")

    target_rank = (percentile_q / 100.0) * total_count
    cumulative_counts = np.cumsum(histogram_counts)

    bin_index = int(np.searchsorted(cumulative_counts, target_rank, side="left"))
    bin_index = int(np.clip(bin_index, 0, len(histogram_counts) - 1))

    count_before = 0 if bin_index == 0 else cumulative_counts[bin_index - 1]
    count_in_bin = int(histogram_counts[bin_index])
    if count_in_bin == 0:
        return float(bin_edges[bin_index])

    fraction_within_bin = (target_rank - count_before) / count_in_bin
    left_edge = float(bin_edges[bin_index])
    right_edge = float(bin_edges[bin_index + 1])
    return float(left_edge + fraction_within_bin * (right_edge - left_edge))


# ==============================
# Core runner (reusable for any column set)
# ==============================
def compute_global_stats_for_columns(
    catalog_path: str,
    path_column_names: Sequence[str],
    root_dir: str,
    out_csv_path: str,
    out_json_path: str,
    bins: int = 4096,
    max_files: int = 0,
    tqdm_mininterval: float = 30.0,
) -> pd.DataFrame:
    """Compute global stats (min, max, mean, std, p1, p2, p98, p99) over 1+ path columns.

    Args:
      catalog_path: CSV file with one or more columns holding TIFF paths.
      path_column_names: Column(s) to read (e.g., ["window_a"] or ["window_a", "window_b"]).
      root_dir: Prefix to apply to relative paths.
      out_csv_path: Where to save the CSV of results.
      out_json_path: Where to save the JSON of results.
      bins: Histogram bin count for percentile estimation.
      max_files: If >0, limit number of paths processed (useful for smoke tests).
      tqdm_mininterval: Minimum seconds between tqdm refreshes (keeps Jupyter happy).

    Returns:
      A DataFrame with one row per band containing global stats.
    """
    # ---- Load catalog & gather paths from the selected columns
    catalog_df: pd.DataFrame = pd.read_csv(catalog_path)
    selected_paths: List[str] = []
    for col in path_column_names:
        if col not in catalog_df.columns:
            raise KeyError(f"Column '{col}' not found in {catalog_path}")
        col_paths = catalog_df[col].dropna().astype(str).tolist()
        selected_paths.extend(col_paths)

    # Optionally limit & remove duplicates to avoid double work
    if max_files and max_files > 0:
        selected_paths = selected_paths[:max_files]
    else:
        # Keep order but de-duplicate
        seen = set()
        uniq_paths = []
        for p in selected_paths:
            if p not in seen:
                seen.add(p)
                uniq_paths.append(p)
        selected_paths = uniq_paths

    # ---- Pass 1: global min/max + exact mean/std (streaming)
    global_min_per_band: Dict[int, float] = defaultdict(lambda: np.inf)
    global_max_per_band: Dict[int, float] = defaultdict(lambda: -np.inf)
    welford_state_per_band: Dict[int, Tuple[int, float, float]] = defaultdict(lambda: (0, 0.0, 0.0))

    detected_band_count: int | None = None
    processed_file_count: int = 0
    missing_file_count: int = 0

    for raw_path in tqdm(selected_paths, desc="Pass 1 (min/max + mean/std)", unit="file", mininterval=tqdm_mininterval):
        tif_path = resolve_path(raw_path, root_dir)
        if not os.path.exists(tif_path):
            missing_file_count += 1
            tqdm.write(f"[WARN] Missing file, skipping: {tif_path}")
            continue

        with rasterio.open(tif_path) as ds:
            band_indices = list(range(1, ds.count + 1))
            if detected_band_count is None:
                detected_band_count = ds.count
            elif ds.count != detected_band_count:
                raise RuntimeError(f"Inconsistent band count: {tif_path} has {ds.count}, expected {detected_band_count}")

            for band_index, values in iterate_valid_blocks(ds, band_indices):
                global_min_per_band[band_index] = min(global_min_per_band[band_index], float(np.min(values)))
                global_max_per_band[band_index] = max(global_max_per_band[band_index], float(np.max(values)))
                welford_state_per_band[band_index] = welford_merge_update(
                    values.astype(np.float64, copy=False), welford_state_per_band[band_index]
                )

        processed_file_count += 1

    if processed_file_count == 0:
        raise RuntimeError("No files processed. Check catalog paths and root_dir.")

    print(
        f"""
Pass 1 complete:
- processed files: {processed_file_count}
- missing files:   {missing_file_count}
- bands detected:  {detected_band_count}
"""
    )
    for band_index in range(1, detected_band_count + 1):
        print(f"Band {band_index}: min={global_min_per_band[band_index]}, max={global_max_per_band[band_index]}")

    # ---- Pass 2: build global histograms → percentiles
    global_histogram_per_band: Dict[int, np.ndarray] = {}
    global_edges_per_band: Dict[int, np.ndarray] = {}

    for band_index in range(1, detected_band_count + 1):
        vmin = global_min_per_band[band_index]
        vmax = global_max_per_band[band_index]
        if not np.isfinite(vmin) or not np.isfinite(vmax):
            vmin, vmax = 0.0, 1.0
        if math.isclose(vmin, vmax):
            vmax = vmin + 1e-9

        bin_edges = np.linspace(vmin, vmax, bins + 1, dtype=np.float64)
        global_edges_per_band[band_index] = bin_edges
        global_histogram_per_band[band_index] = np.zeros(bins, dtype=np.int64)

    for raw_path in tqdm(selected_paths, desc="Pass 2 (global histograms)", unit="file", mininterval=tqdm_mininterval):
        tif_path = resolve_path(raw_path, root_dir)
        if not os.path.exists(tif_path):
            continue

        with rasterio.open(tif_path) as ds:
            band_indices = list(range(1, ds.count + 1))
            for band_index, values in iterate_valid_blocks(ds, band_indices):
                bin_edges = global_edges_per_band[band_index]
                left_edge, right_edge = bin_edges[0], bin_edges[-1]
                values = values[(values >= left_edge) & (values <= right_edge)]
                if values.size == 0:
                    continue
                hist_counts, _ = np.histogram(values, bins=bin_edges)
                global_histogram_per_band[band_index] += hist_counts

    print("Pass 2 complete.")

    # ---- Finalize & save
    requested_percentiles: List[int] = [1, 2, 98, 99]
    result_records: List[Dict[str, float | int]] = []

    for band_index in range(1, detected_band_count + 1):
        n_obs, mean_value, M2_value = welford_state_per_band[band_index]
        std_value = math.sqrt(M2_value / (n_obs - 1)) if n_obs > 1 else float("nan")
        bin_edges = global_edges_per_band[band_index]
        band_hist = global_histogram_per_band[band_index]

        percentile_values = {
            f"p{q}": percentile_from_histogram(band_hist, bin_edges, q) for q in requested_percentiles
        }

        record = {
            "band": band_index,
            "count": int(n_obs),
            "mean": mean_value,
            "std": std_value,
            "min": float(global_min_per_band[band_index]),
            "max": float(global_max_per_band[band_index]),
            **percentile_values,
        }
        result_records.append(record)

    global_stats_df: pd.DataFrame = pd.DataFrame(result_records).sort_values("band").reset_index(drop=True)
    display(global_stats_df)

    global_stats_df.to_csv(out_csv_path, index=False)
    with open(out_json_path, "w") as f_out:
        json.dump(result_records, f_out, indent=2)

    print(f"Wrote: {out_csv_path} and {out_json_path}")
    return global_stats_df


# # ==============================
# # RUN 1 — FTW window_a only
# # ==============================
# stats_window_a = compute_global_stats_for_columns(
#     catalog_path=ftw_catalog_path,
#     path_column_names=["window_a"],
#     root_dir=root_data_dir,
#     out_csv_path="/home/airg/gessuman/projects/ftw-mappingafrica-integration/data/global_band_stats_ftw_window_a.csv",
#     out_json_path="/home/airg/gessuman/projects/ftw-mappingafrica-integration/data/global_band_stats_ftw_window_a.json",
#     bins=histogram_bin_count,
#     max_files=max_files_to_process,
# )

# ==============================
# RUN 2 — FTW window_a + window_b combined
# ==============================
stats_window_ab = compute_global_stats_for_columns(
    catalog_path=ftw_catalog_path,
    path_column_names=["window_a", "window_b"],
    root_dir=root_data_dir,
    out_csv_path="/home/airg/gessuman/projects/ftw-mappingafrica-integration/data/global_band_stats_ftw_window_ab.csv",
    out_json_path="/home/airg/gessuman/projects/ftw-mappingafrica-integration/data/global_band_stats_ftw_window_ab.json",
    bins=histogram_bin_count,
    max_files=max_files_to_process,
)


Pass 1 (min/max + mean/std):   1%|▍                                           | 1231/140968 [01:30<2:53:51, 13.40file/s]